## Sixth step

In `sixth_step.ipynb`, we will use the records filtered (or not) in `fifth_step.ipynb` and filter the scientific names of the herbarium specimens. Much like for taxonomists, the scientific name of the plants is crucial for properly training an artificial intelligence model. Typographical errors and synonyms identified by The Leipzig Catalogue of Vascular Plants (LCVP) will be standardized to a single name.

Following the standart from the previous steps, the new field will be named `scientificname_upd`, as the original field is `scientific_name`. We will also include `plant_status`, a new field, to identify names that were already correct (*accepted*), synonyms that were updated (*synonym*), and names not found in the LCVP database (*unresolved*).

If you do not wish to perform this filtering, continue using the `scientificname` field instead of `scientificname_upd` for subsequent filtering steps.

In [ ]:
from library import *

specieslink, db_config = configure()

In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

requirement = "country_upd"
requirement2 = "stateprovince_upd"
requirement3 = "barcode_upd"
requirement4 = "identifiedby_upd"
column = "scientificname"
table = "biodiversity_records"

sql = f"""SELECT {column} FROM {table} WHERE {column} IS NOT NULL AND {requirement} IS NOT NULL AND {requirement2} IS NOT NULL AND {requirement3} IS NOT NULL AND {requirement4} IS NOT NULL GROUP BY {column}"""

cursor.execute(sql)
results = cursor.fetchall()

cursor.close()
conn.close()

nomes = [r[0] for r in results if r[0]]

df = pd.DataFrame({"scientificname": nomes})

csv_buffer = StringIO()
df.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

In [ ]:
table = "biodiversity_records"
column = input("enter the name of the PRE-CREATED column that will store the NEW scientific name of the plants:").strip()
status = input("enter the name of the PRE-CREATED column that will store the status of the CURRENT scientific name of the plants: ").strip()
max_distance = input("specify the maximum margin of error for grammatical errors and the like (standard 0.1): ").strip()
if max_distance:
    try:
        max_distance = float(max_distance)
    except ValueError:
        print("invalid margin error value. using standard value of 0.1\n")
        max_distance = 0.1
else:
    max_distance = 0.1

    try:
        conn = mysql_conn.connect(**db_config)
        cursor = conn.cursor()

        for col in [column, status]:
                try:
                    cursor.execute(f"ALTER TABLE {table} ADD COLUMN {col} TEXT")
                    print(f"field '{col}' created with succes in table '{table}")
                except Exception:
                    pass 
        conn.commit()
        cursor.close()
        conn.close()
    except Exception as e:
        print(f"error handling columns: {e}")

csv_buffer.seek(0)
lines = csv_buffer.read().splitlines()

lines = [line.strip() for line in lines if line.strip()]

if lines and lines[0].lower() in ("scientificname", "scientific_name"):
    lines = lines[1:]

if not lines:
    raise RuntimeError("CSV doesn't contain valid scientific names after filtering")

csv_content = "\n".join(lines) + "\n"

with tempfile.NamedTemporaryFile(
    mode="w",
    suffix=".csv",
    delete=False,
    encoding="utf-8"
) as tmp:
    tmp.write(csv_content)
    tmp_path = tmp.name


print(f"executing synonym.perform_lcvp_fuzzy_search_per_line(--csv {tmp_path} --table {table} --column {column} --status {status})...\n")

fuzzy_line = perform_lcvp_fuzzy_search_per_line(
    csv_file=tmp_path, db_config=db_config, table=table, column=column, specieslink=specieslink, status=status, max_distance=max_distance
)
if fuzzy_line:
    print("\nverifying with LCVP...")

os.remove(tmp_path)

In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

sql = """
SELECT
    COUNT(*) AS total_records,
    SUM(country_upd IS NOT NULL) AS count_country,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL) AS count_country_n_state,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL AND barcode_upd IS NOT NULL) as count_country_state_n_barcode,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL AND barcode_upd IS NOT NULL AND identifiedby_upd IS NOT NULL) as count_country_state_barcode_n_taxonomist
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL AND barcode_upd IS NOT NULL AND identifiedby_upd IS NOT NULL AND status_plantas = "accepted" OR status_plantas = "synonym") as count_country_state_barcode_taxonomist_name
FROM biodiversity_records
"""

cursor.execute(sql)
total_records, records_country, records_country_state, records_country_state_barcode, records_country_state_barcode_taxonomist, records_country_state_barcode_taxonomist_name = cursor.fetchone()

cursor.close()
conn.close()

In [ ]:
plt.figure(figsize=(6, 4))

x = [0]

plt.bar(
    x,
    [total_records],
    alpha=0.5,
    width=0.15,
    label='total'
)

plt.bar(
    x,
    [records_country],
    alpha=0.5,
    width=0.15,
    label='country_att'
)

plt.bar(
    x,
    [records_country_state],
    width=0.15,
    label='country_att and stateprovince_att'
)

plt.bar(
    x,
    [records_country_state_barcode],
    width=0.15,
    label='country_att, stateprovince_att and barcode_att'
)

plt.bar(
    x,
    [records_country_state_barcode_taxonomist],
    width=0.15,
    label='country_att, stateprovince_att, barcode_att and identifiedby_att'
)

plt.bar(
    x,
    [records_country_state_barcode_taxonomist_name],
    width=0.15,
    label='country_att, stateprovince_att, barcode_att, identifiedby_att and scientificname_att'
)

plt.xlim(-0.2, 0.3)
plt.ylabel('total records')
plt.title('sample of records to be used')
plt.legend()

plt.text(0, total_records, str(total_records), ha='center', va='bottom')
plt.text(0, records_country, str(records_country), ha='center', va='bottom')
plt.text(0, records_country_state, str(records_country_state), ha='center', va='bottom')
plt.text(0, records_country_state_barcode, str(records_country_state_barcode), ha='center', va='bottom')
plt.text(0, records_country_state_barcode, str(records_country_state_barcode_taxonomist), ha='center', va='bottom')
plt.text(0, records_country_state_barcode, str(records_country_state_barcode_taxonomist_name), ha='center', va='bottom')

plt.tight_layout()
plt.show()